# ModernBERT — CEFR classification (fine-tuning, val)

Fine-tunes ModernBERT-large on Write & Improve (A2–C1+, 8 levels), nominal
cross-entropy head only. Uses the same `metrics.py` as the other conditions.

Ends by writing `dev_predictions.csv` in the same column layout as the Gemma 4
notebooks (`p_pred`, per-class probabilities, margin, entropy, expected level),
so the results notebook can read all four conditions identically.

In [3]:
# %%
import os, json, csv
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, Trainer, TrainingArguments,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput

from metrics import evaluate_predictions, LABEL_NAMES

LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]
LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}
NUM_CLASSES = len(LEVELS)  # 8


@dataclass
class CFG:
    model_id: str = "answerdotai/ModernBERT-large"    # ~395M params
    train_path: str = "data/train.jsonl"
    eval_path: str  = "data/val.jsonl"
    text_col: str = "text"
    label_col: str = "label"
    max_length: int = 1024
    epochs: float = 4.0
    batch_size: int = 8
    grad_accum: int = 1
    lr: float = 2e-5
    weight_decay: float = 0.01
    bf16: bool = True
    use_lora: bool = False
    lora_r: int = 16
    output_dir: str = "runs/modernbert_nominal"
    seed: int = 42

cfg = CFG()

## Data

In [4]:
def read_rows(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".jsonl":
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    yield json.loads(line)
    elif ext == ".json":
        with open(path, encoding="utf-8") as f:
            yield from json.load(f)
    else:  # csv / tsv
        delim = "\t" if ext == ".tsv" else ","
        with open(path, encoding="utf-8", newline="") as f:
            yield from csv.DictReader(f, delimiter=delim)


class CEFRDataset(Dataset):
    def __init__(self, path, tokenizer, cfg):
        self.ex, skipped = [], 0
        for row in read_rows(path):
            lab = str(row[cfg.label_col]).strip().upper()
            if lab not in LABEL2ID:
                skipped += 1
                continue
            self.ex.append((str(row[cfg.text_col]), LABEL2ID[lab]))
        print(f"{path}: kept {len(self.ex)}, skipped {skipped} out-of-scope")
        self.tok, self.max_length = tokenizer, cfg.max_length

    def __len__(self):
        return len(self.ex)

    def __getitem__(self, i):
        text, label = self.ex[i]
        enc = self.tok(text, truncation=True, max_length=self.max_length)
        enc["labels"] = label
        return enc

In [5]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
train_ds = CEFRDataset(cfg.train_path, tokenizer, cfg)
eval_ds  = CEFRDataset(cfg.eval_path,  tokenizer, cfg)
collator = DataCollatorWithPadding(tokenizer)

data/train.jsonl: kept 3797, skipped 0 out-of-scope
data/val.jsonl: kept 599, skipped 0 out-of-scope


## Model

Mean-pool encoder outputs (attention-mask aware), then project through a
single linear head: `hidden → 8`.

In [6]:
class ModernBERTClassifier(nn.Module):
    def __init__(self, model_id, num_classes=8, dropout=0.1, dtype=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id, torch_dtype=dtype)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, num_classes)
        self.head.to(torch.float32)  # stable logits even with bf16 backbone

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(h.dtype)
            pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        else:
            pooled = h.mean(1)
        logits = self.head(self.dropout(pooled).to(torch.float32))
        return SequenceClassifierOutput(logits=logits)

In [7]:
dtype = torch.bfloat16 if cfg.bf16 else torch.float32
model = ModernBERTClassifier(cfg.model_id, num_classes=NUM_CLASSES, dtype=dtype)

if cfg.use_lora:
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(
        r=cfg.lora_r, lora_alpha=2 * cfg.lora_r, lora_dropout=0.05,
        target_modules=["Wqkv"],       # ModernBERT fuses QKV into one projection
        modules_to_save=["head"],
    ))
    model.print_trainable_parameters()

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params: {total:,} total, {train:,} trainable")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


params: 394,789,896 total, 394,789,896 trainable


## Trainer

In [8]:
class NominalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels.long())
        return (loss, out) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.pop("labels")
        with torch.no_grad():
            out = model(**inputs)
            loss = torch.nn.functional.cross_entropy(out.logits, labels.long())
        if prediction_loss_only:
            return (loss, None, None)
        return (loss, out.logits, labels)


def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.asarray(logits).argmax(-1)
    return evaluate_predictions(eval_pred.label_ids, preds)

In [9]:
steps_per_epoch = len(train_ds) // (cfg.batch_size * cfg.grad_accum)
total_steps = int(steps_per_epoch * cfg.epochs)
warmup_steps = int(0.06 * total_steps)

args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    weight_decay=cfg.weight_decay,
    warmup_steps=warmup_steps,
    bf16=cfg.bf16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_qwk",
    greater_is_better=True,
    save_total_limit=2,
    seed=cfg.seed,
    report_to="none",
)

trainer = NominalTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator, compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Qwk,Mae,Accuracy,Adjacent Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,1.380839,1.336716,0.839233,0.641068,0.467446,0.909850,0.429498,0.339189,0.289508,0.511428,0.467446,0.407171
2,1.223443,1.239728,0.835377,0.617696,0.475793,0.918197,0.339872,0.343733,0.314147,0.452700,0.475793,0.440310
3,1.217121,1.201667,0.852746,0.579299,0.500835,0.934891,0.353066,0.364888,0.343542,0.475486,0.500835,0.465433
4,1.182209,1.198838,0.849400,0.590985,0.489149,0.934891,0.338957,0.354055,0.332724,0.459812,0.489149,0.455672


TrainOutput(global_step=1900, training_loss=1.3612672564857884, metrics={'train_runtime': 175.8945, 'train_samples_per_second': 86.347, 'train_steps_per_second': 10.802, 'total_flos': 0.0, 'train_loss': 1.3612672564857884, 'epoch': 4.0})

## Val evaluation — metrics + predictions

In [11]:
dev_results = trainer.evaluate()
print("best dev:", json.dumps(dev_results, indent=2))

dev_out = trainer.predict(eval_ds)
dev_logits = dev_out.predictions
if isinstance(dev_logits, tuple):
    dev_logits = dev_logits[0]
dev_logits_t = torch.as_tensor(np.asarray(dev_logits, dtype=np.float32))
dev_preds = dev_logits_t.argmax(-1).numpy()
dev_gold = dev_out.label_ids

probs = torch.softmax(dev_logits_t, dim=-1).numpy()
print("logits shape:", tuple(dev_logits_t.shape))
print("prob rows sum to 1:", np.allclose(probs.sum(1), 1.0, atol=1e-4))

Training Loss,Validation Loss,Epoch,Qwk,Mae,Accuracy,Adjacent Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1.182209,1.201667,4,0.852746,0.579299,0.500835,0.934891,0.353066,0.364888,0.343542,0.475486,0.500835,0.465433


best dev: {
  "eval_loss": 1.2016667127609253,
  "eval_qwk": 0.852746364372746,
  "eval_mae": 0.5792988313856428,
  "eval_accuracy": 0.5008347245409015,
  "eval_adjacent_accuracy": 0.9348914858096828,
  "eval_precision_macro": 0.3530659155659156,
  "eval_recall_macro": 0.3648883148969532,
  "eval_f1_macro": 0.34354174360592377,
  "eval_precision_weighted": 0.4754862632575488,
  "eval_recall_weighted": 0.5008347245409015,
  "eval_f1_weighted": 0.46543328764746117
}


logits shape: (599, 8)
prob rows sum to 1: True


In [12]:
print("dev pred distribution:", Counter(dev_preds.tolist()))
print("dev gold distribution:", Counter(dev_gold.tolist()))

dev pred distribution: Counter({1: 198, 3: 153, 4: 104, 2: 66, 5: 54, 6: 24})
dev gold distribution: Counter({1: 136, 2: 128, 4: 110, 3: 95, 5: 49, 6: 37, 0: 30, 7: 14})


## Confusion matrix

In [14]:
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

counts = confusion_matrix(dev_gold, dev_preds, labels=list(range(NUM_CLASSES)))
row_sums = counts.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    norm = np.nan_to_num(np.divide(counts, row_sums, where=row_sums != 0))

annot = np.empty_like(counts, dtype=object)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        annot[i, j] = (f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"
                       if counts[i, j] else "")

fig = go.Figure(go.Heatmap(
    z=norm, x=LEVELS, y=LEVELS,
    text=annot, texttemplate="%{text}",
    hoverongaps=False,
    colorscale="Greys", zmin=0, zmax=1,
    colorbar=dict(title="Row-norm"),
))
fig.update_layout(
    title="ModernBERT-large nominal — dev confusion matrix",
    xaxis=dict(title="Predicted"),
    yaxis=dict(title="True", autorange="reversed"),
    font=dict(family="Arial", size=14, color="black"),
    width=600, height=550,
)
fig.show()

## Predictions CSV

Same column layout as the other three conditions (Gemma 4 prompted, Gemma 4
LoRA, T5Gemma2), so the ECE/AUROC/calibration cells in the results notebook
can read this directly.

In [15]:
srt = np.sort(probs, axis=1)[:, ::-1]
rows = []
for i, (g, p) in enumerate(zip(dev_gold, dev_preds)):
    g, p = int(g), int(p)
    lo, hi = max(0, p - 1), min(NUM_CLASSES - 1, p + 1)
    rows.append(dict(
        gold=g, pred=p, gold_label=LEVELS[g], pred_label=LEVELS[p],
        correct=int(g == p), adjacent=int(abs(g - p) <= 1),
        p_pred=float(probs[i, p]),
        p_adjacent=float(probs[i, lo:hi + 1].sum()),
        margin=float(srt[i, 0] - srt[i, 1]),
        entropy=float(-(probs[i] * np.log(probs[i] + 1e-12)).sum()),
        exp_level=float((probs[i] * np.arange(NUM_CLASSES)).sum()),
        **{f"p_{lvl}": float(probs[i, j]) for j, lvl in enumerate(LEVELS)},
    ))

dev_df = pd.DataFrame(rows)

os.makedirs(cfg.output_dir, exist_ok=True)
with open(os.path.join(cfg.output_dir, "dev_metrics.json"), "w") as f:
    json.dump(dev_results, f, indent=2)
dev_df.to_csv(os.path.join(cfg.output_dir, "dev_predictions.csv"), index=False)

print(f"Saved to {cfg.output_dir}/dev_metrics.json and dev_predictions.csv")

Saved to runs/modernbert_nominal/dev_metrics.json and dev_predictions.csv


## Does confidence separate correct from wrong?
### 0.5 is chance, >0.7 is what I want to see

In [16]:
from sklearn.metrics import roc_auc_score


def ece(d, n_bins=10):
    b = pd.cut(d["p_pred"], np.linspace(0, 1, n_bins + 1))
    g = d.groupby(b, observed=True)
    gap = (g["correct"].mean() - g["p_pred"].mean()).abs()
    return float((gap * g.size()).sum() / len(d))


print(f"mean p_pred  {dev_df['p_pred'].mean():.3f}")
print(f"mean entropy {dev_df['entropy'].mean():.3f}  (max {np.log(NUM_CLASSES):.3f})")
print(f"ECE          {ece(dev_df):.4f}")
print()
for col, s in [("p_pred", dev_df["p_pred"]), ("margin", dev_df["margin"]),
               ("entropy", -dev_df["entropy"]), ("p_adjacent", dev_df["p_adjacent"])]:
    print(f"{col:12s} exact {roc_auc_score(dev_df['correct'], s):.3f}   "
          f"adjacent {roc_auc_score(dev_df['adjacent'], s):.3f}")
print()
print(dev_df.groupby("correct")[["p_pred", "margin", "entropy"]].mean().round(3))

mean p_pred  0.479
mean entropy 1.300  (max 2.079)
ECE          0.0431

p_pred       exact 0.582   adjacent 0.667
margin       exact 0.562   adjacent 0.613
entropy      exact 0.591   adjacent 0.693
p_adjacent   exact 0.616   adjacent 0.716

         p_pred  margin  entropy
correct                         
0         0.467   0.181    1.323
1         0.491   0.208    1.277


In [20]:
edges = np.arange(0.0, 1.01, 0.1)
BANDS = list(zip(edges[:-1], edges[1:],
                 [f"{lo:.1f}\u2013{hi:.1f}" for lo, hi in zip(edges[:-1], edges[1:])]))

n = len(dev_df)
band_rows = []
for lo, hi, name in reversed(BANDS):
    s = dev_df[(dev_df["p_pred"] >= lo) & (dev_df["p_pred"] < hi)]
    band_rows.append(dict(
        conf=name, preds=len(s), share=f"{len(s)/n*100:.0f}%",
        acc=round(s["correct"].mean(), 2) if len(s) else None,
        adj=round(s["adjacent"].mean(), 2) if len(s) else None,
    ))

print(pd.DataFrame(band_rows).to_string(index=False))

   conf  preds share  acc  adj
0.9–1.0      0    0%  NaN  NaN
0.8–0.9      0    0%  NaN  NaN
0.7–0.8      3    1% 0.33 0.67
0.6–0.7     62   10% 0.55 0.95
0.5–0.6    181   30% 0.57 0.98
0.4–0.5    229   38% 0.51 0.94
0.3–0.4    119   20% 0.36 0.87
0.2–0.3      5    1% 0.40 0.60
0.1–0.2      0    0%  NaN  NaN
0.0–0.1      0    0%  NaN  NaN
